# Лекция 5. PGD и Carlini-Wagner атаки

Демонстрация: реализация PGD и упрощённой C&W-атаки, сравнение с FGSM.

## 1. Обучение базовой модели

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

import torchvision
import torchvision.transforms as T

transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=True)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc = nn.Linear(32*7*7,10)
    def forward(self,x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x,2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x,2)
        x = x.view(x.size(0),-1)
        return self.fc(x)

model = SimpleCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for i,(xb,yb) in enumerate(train_loader):
    opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
    if i>=300: break
print("Базовая модель обучена, loss:", loss.item())


Базовая модель обучена, loss: 0.20680689811706543


## 2. Реализация PGD

In [2]:

def pgd_attack(model, x, y, eps, alpha, iters):
    x_orig = x.clone().detach()
    x_adv = x_orig + torch.empty_like(x_orig).uniform_(-eps, eps)
    x_adv = torch.clamp(x_adv, 0, 1).detach()
    for _ in range(iters):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha*grad.sign()
        x_adv = torch.max(torch.min(x_adv, x_orig+eps), x_orig-eps)
        x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

x, y = next(iter(test_loader))
x_pgd = pgd_attack(model, x, y, eps=0.2, alpha=0.02, iters=20)
print("Истинный класс:", y.item(), "Предсказание после PGD:", model(x_pgd).argmax(1).item())


Истинный класс: 4 Предсказание после PGD: 9


## 3. Упрощённая C&W атака (L2, оптимизационная)

In [3]:

def cw_attack(model, x, y, c=1.0, iters=100, lr=0.01):
    x_orig = x.clone().detach()
    w = torch.zeros_like(x_orig, requires_grad=True)
    opt = torch.optim.Adam([w], lr=lr)
    for _ in range(iters):
        x_adv = torch.clamp(x_orig + w, 0, 1)
        out = model(x_adv)
        target_logit = out[0, y.item()]
        other_logit = out[0][torch.arange(10) != y.item()].max()
        loss_adv = torch.clamp(target_logit - other_logit, min=-0.0)
        loss = (w**2).sum() + c*loss_adv
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.clamp(x_orig + w, 0, 1).detach()

x_cw = cw_attack(model, x, y, c=5.0, iters=150)
l2_pgd = (x_pgd-x).pow(2).sum().sqrt().item()
l2_cw = (x_cw-x).pow(2).sum().sqrt().item()
print(f"PGD: pred={model(x_pgd).argmax(1).item()}, L2 norm={l2_pgd:.3f}")
print(f"C&W: pred={model(x_cw).argmax(1).item()}, L2 norm={l2_cw:.3f}")
print("Вывод: C&W обычно находит возмущение с меньшей нормой при успешной атаке")


PGD: pred=9, L2 norm=3.604
C&W: pred=4, L2 norm=1.593
Вывод: C&W обычно находит возмущение с меньшей нормой при успешной атаке
